In [3]:
import re
import os
import pandas as pd
import numpy as np
from Bio import SeqIO
from datetime import datetime
from plotnine import *
# import geopandas as gpd

In [4]:
# Goal: Generate metadata and FASTA files for a dataset that just corresponds to live/harvested detections

In [5]:
# Upload the detections data

FILE1 = "./metadata/detections_data/combinedNAm_detections_2021-2024.csv"

full_NAm_detections = pd.read_csv(FILE1, sep = ",")

In [6]:
print(full_NAm_detections.columns)

Index(['Unnamed: 0', 'Collection Date', 'Location', 'Avian_Host', 'Country',
       'Flyway', 'Latitude_Group', 'Host_Order', 'Host_Broad', 'Sample_Method',
       'Corrected_Avian_Host'],
      dtype='object')


In [7]:
# Upload the sequencing metadata

FILE2 = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_ha_detectionsgeodata.tsv"

metadata = pd.read_csv(FILE2, sep = '\t')

In [8]:
print(metadata.columns)
print(len(metadata))

Index(['Unnamed: 0.1', 'Unnamed: 0', 'SRA_Accession', 'strain_segment',
       'Genome ID', 'H5 Clade', 'SRA Accession', 'strain', 'h5_clade',
       'source_merge2', 'bvbrc_assembled', 'strain_segment_updated',
       'genbank_acc', 'bp_length', 'segment_no', 'nonambig_date', 'date',
       'strain_title', 'division', 'host_fromstrain', 'bird_order',
       'domesticstatus_combined', 'host_correction', 'NAflyway',
       'state_lat-long', 'strain_title_updated', 'year_only', 'month_only',
       'county', 'city', 'county_lat-long', 'city_lat-long', 'lat_long_to_use',
       'lat_long_source', 'lat', 'long'],
      dtype='object')
6737


In [9]:
# Step one: label each sequence that matches a detection with the sampling method

In [10]:
# Function: what to do when a matching row(s) exists 
# This function is intended as a nesting function inside the next function

def filling_rows_samplingmethod(metadata_sampling, idx, matching_rows):
    
    unique_methods_in_match = set(matching_rows["Sample_Method"].tolist()) # If there are more than one option for sampling method
    
    # if matching rows exist, this function will follow a specific set of instructions
    if len(unique_methods_in_match) == 1: 
        # If there is one matching method, add the method
        metadata_sampling.at[idx, "detection_samplingmethod"] = matching_rows["Sample_Method"].iloc[0]
                    
    # If there is more than one "county" match in matching rows
    elif len(unique_methods_in_match) > 1:
        metadata_sampling.at[idx, "detection_samplingmethod"] = "unknown"

    # If no matching rows are found, set county to "unknown"
    else:
        print(f"Unexpected case (no matching methods) at index {idx}")

    return(metadata_sampling)

In [11]:
# Function: how to detect and fill matching rows

def matching_datehostdivision(metadata, detections):

    metadata_sampling = metadata.copy()
    
    for idx, row in metadata_sampling.iterrows():
        # Find rows in detections_H5N1_newcolumns that match the current row's values
        matching_rows = detections[
            (detections["Collection Date"] == row["date"]) &
            (detections["Location"] == row["division"]) &
            (detections["Corrected_Avian_Host"] == row["host_fromstrain"])
        ]
        
        # If matching rows exist, assign the corresponding county or value
        if not matching_rows.empty:
            metadata_sampling = filling_rows_samplingmethod(metadata_sampling, idx, matching_rows)
        else: 
            metadata_sampling.at[idx, "detection_samplingmethod"] = "unknown" 

    return(metadata_sampling)

In [12]:
metadata_with_sampling_method = matching_datehostdivision(metadata, full_NAm_detections)

In [13]:
print(metadata_with_sampling_method["detection_samplingmethod"].value_counts())
print(len(metadata_with_sampling_method))

detection_samplingmethod
Dead                     3984
unknown                  1456
Agency/Hunter_Harvest     686
Live                      275
Euthanized                  9
Name: count, dtype: int64
6737


In [14]:
# Step two: subset to just live and harvested detections

In [15]:
sampling = ["Agency/Hunter_Harvest", "Live"]

metadata_liveharvest = metadata_with_sampling_method[metadata_with_sampling_method["detection_samplingmethod"].isin(sampling)]
print(len(metadata_liveharvest))

961


In [16]:
metadata_with_sampling_method.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_ha_live-harvest-only.tsv", sep = '\t')

In [ ]:
# Step three: make a FASTA with corresponding sequences

In [136]:
#Function: Read in the FASTA as a dataframe

def fasta_reader(path_to_fasta, output_name):
    fasta_data = []
    FASTANAME = path_to_fasta
        
    with open(FASTANAME) as f:
        header = ""
        sequence = ""
        for line in f:
            if line.startswith(">"):
                if header != "":
                    fasta_data.append({"header": header, "sequence": sequence})
                header = line.strip() 
                sequence = ""
            else:
                sequence += line.strip()
        fasta_data.append({"header": header, "sequence": sequence}) #last line 
        
    globals()[output_name] = pd.DataFrame(fasta_data)

    return

In [137]:
# Function to write a new fasta with just the sequences in the metadata
# Adapted from Maria

def fasta_writer(path, filename, df):
            
    try:  
        os.mkdir(path)

    except OSError as error:
        pass

    with open(f"{path}{filename}", "w") as f:
        for index, row in df.iterrows():
            f.write(f"{row['header']}\n")
            f.write(f"{row['sequence']}\n")

In [138]:
# Function: remove the carrot from the header

def header_to_strain(df):
    df["strain"] = df["header"].str.replace(">","")

    return(df)

In [139]:
# Function: parse to include

def parse_toinclude(objects_list, file_to_parse):
    # Define the list as the objects to include
    to_include = objects_list

    # Filter rows where the 'taxon' value is in the strains_set
    parsed = file_to_parse[file_to_parse["strain"].isin(to_include)]

    return parsed

In [141]:
# List the strain names

list_of_strains = metadata_with_sampling_method["strain"].to_list()

fasta_reader("./raw_data/ncbi-bvbrc-gisaid_from-meta_ha.fasta", "ha_fasta")

ha_fasta_strain = header_to_strain(ha_fasta)

parsed_fasta = parse_toinclude(list_of_strains, ha_fasta_strain)

In [142]:
print(len(parsed_fasta))

1078


In [2]:
#fasta_writer("./raw_data/", "ncbi-bvbrc-gisaid_live-harvest-only_ha.fasta", parsed_fasta)